In [ ]:

from pyspark.sql import functions as F
from lakehouse_spark import SparkSessionContext
from lakehouse_publish import publish_table

TABLE = "lakehouse.stage.player_match_heroes"

stage_sql = """
WITH matches_ok AS (
    SELECT
        match_id,
        start_time,
        duration,
        radiant_win,
        game_mode
    FROM lakehouse.raw.pg__matches
    WHERE duration >= 600
      AND radiant_win IS NOT NULL
    ORDER BY match_id
),

team_counts AS (
    SELECT
        mp.match_id,
        SUM(CASE WHEN mp.is_radiant = TRUE THEN 1 ELSE 0 END) AS n_radiant,
        SUM(CASE WHEN mp.is_radiant = FALSE THEN 1 ELSE 0 END) AS n_dire
    FROM lakehouse.raw.pg__match_players mp
    INNER JOIN matches_ok m
        ON m.match_id = mp.match_id
    GROUP BY mp.match_id
),

players_ok AS (
    SELECT
        mp.match_id,
        mp.player_slot,
        mp.hero_id,
        mp.is_radiant,
        mp.win,
        mp.account_id
    FROM lakehouse.raw.pg__match_players mp
    INNER JOIN team_counts tc
        ON tc.match_id = mp.match_id
    WHERE tc.n_radiant = 5
      AND tc.n_dire = 5
      AND mp.account_id IS NOT NULL
      AND mp.account_id > 0
      AND mp.account_id <> 4294967295
      AND mp.hero_id IS NOT NULL
      AND mp.hero_id > 0
      AND mp.win IS NOT NULL
)

SELECT
    po.match_id,
    m.start_time,
    m.duration,
    m.game_mode,
    m.radiant_win,
    po.player_slot,
    po.is_radiant,
    po.hero_id,
    h.localized_name AS hero_name,
    h.primary_attr AS hero_attr,
    po.account_id,
    po.win
FROM players_ok po
INNER JOIN matches_ok m
    ON m.match_id = po.match_id
INNER JOIN lakehouse.raw.pg__heroes h
    ON h.hero_id = po.hero_id
"""

with SparkSessionContext("stage-player-match-heroes") as ctx:
    spark = ctx

    #Создаём namespace
    spark.sql("CREATE NAMESPACE IF NOT EXISTS lakehouse.stage")

    #Выполняем SQL
    with ctx.lineage(
        "stage-player-match-heroes",
        inputs=[
            "raw.pg__matches",
            "raw.pg__match_players",
            "raw.pg__heroes",
        ],
        output="stage.player_match_heroes",
        description="STAGE: одна строка на игрока в матче (5v5, валидные ключи).",
    ):
        df = spark.sql(stage_sql).persist()

        rows = df.count()
        print(f"Строк получено из SQL: {rows}")

        if rows == 0:
            raise ValueError("SQL вернул 0 строк. Запись таблицы отменена.")

        #Проверяем DataFrame перед записью
        print("\n--- SCHEMA ---")
        df.printSchema()

        print("\n--- DATA PREVIEW ---")
        df.show(10, truncate=False)

        #Записываем таблицу в Iceberg
        print(f"\nЗаписываем таблицу: {TABLE}")

        df.writeTo(TABLE).using("iceberg").createOrReplace()

        print("Запись в Iceberg завершена.")

        #Обновляем метаданные таблицы в Spark
        spark.catalog.refreshTable(TABLE)

        #Проверяем, что таблица действительно существует
        print("\n--- TABLES IN lakehouse.stage ---")
        spark.sql("SHOW TABLES IN lakehouse.stage").show(truncate=False)

        #Читаем таблицу обратно из Iceberg
        result_df = spark.table(TABLE)

        print("\n--- ICEBERG TABLE COUNT ---")
        print("Количество строк:", result_df.count())

        print("\n--- ICEBERG TABLE PREVIEW ---")
        result_df.show(10, truncate=False)

        #Проверяем количество матчей
        print("\n--- MATCH CHECK ---")
        result_df.select(
            F.count("*").alias("rows"),
            F.countDistinct("match_id").alias("unique_matches"),
            F.countDistinct("hero_id").alias("unique_heroes"),
        ).show()

        #Проверяем Iceberg snapshots
        print("\n--- ICEBERG SNAPSHOTS ---")
        spark.sql(f"""
            SELECT snapshot_id, committed_at, operation
            FROM {TABLE}.snapshots
            ORDER BY committed_at DESC
            LIMIT 3
        """).show(truncate=False)

        #Публикуем таблицу через Lakehouse
        result = publish_table(
            "stage",
            "player_match_heroes",
            result_df,
            description="STAGE: игрок–матч–герой для дальнейшего MART",
            dependencies=[
                "raw/pg__matches",
                "raw/pg__match_players",
                "raw/pg__heroes",
            ],
        )

        print("\n--- PUBLISH RESULT ---")
        print(result)

        df.unpersist(blocking=True)
